# 订单核查规则 v2：版本与并集计数

v1 六条规则保留为课堂基线，v2 增加审批相关两条规则。输入订单全部保留，业务指标继续按各自有效样本条件计算。

本 Notebook 独立读取原始数据，不改动前两份学习记录。

In [13]:
from pathlib import Path
import sys
import importlib
import pandas as pd

project_dir = Path(r'D:\数据分析\olist-operations-analytics')
if str(project_dir / 'src') not in sys.path:
    sys.path.insert(0, str(project_dir / 'src'))

# 重新加载更新后的函数文件，避免内核保留旧函数。
import orders_cleaning
orders_cleaning = importlib.reload(orders_cleaning)
print('当前 Python：', sys.executable)

当前 Python： e:\Anaconda3\envs\myenv\python.exe


In [14]:
raw_orders = pd.read_csv(
    project_dir.parent / 'archive' / 'olist_orders_dataset.csv',
    dtype=str, encoding='utf-8-sig', keep_default_na=False, na_values=['']
)

orders_v1 = orders_cleaning.clean_orders(raw_orders, rules_version='v1')
orders_v2 = orders_cleaning.clean_orders(raw_orders, rules_version='v2')
metrics_v1 = orders_cleaning.summarize_metrics(orders_v1, rules_version='v1')
metrics_v2 = orders_cleaning.summarize_metrics(orders_v2, rules_version='v2')

In [15]:
# 比较两个规则版本：前两项核查统计变化，业务样本条件保持原定义。
keys = ['order_rows', 'review_rule_hits', 'review_orders',
        'valid_transit_orders', 'valid_delay_orders', 'late_orders']
pd.DataFrame({
    'v1': {key: metrics_v1[key] for key in keys},
    'v2': {key: metrics_v2[key] for key in keys}
})

,v1,v2
order_rows,99441,99441
review_rule_hits,219,1578
review_orders,218,1411
valid_transit_orders,96281,96281
valid_delay_orders,96470,96470
late_orders,6534,6534


In [16]:
flag_cols_v2 = orders_cleaning.get_review_flag_cols('v2')
orders_v2[flag_cols_v2].sum()

flag_delivered_missing_date           8
flag_delivered_missing_approval      14
flag_delivered_missing_carrier        2
flag_carrier_before_purchase        166
flag_delivery_before_carrier         23
flag_canceled_has_delivery_date       6
flag_approval_before_purchase         0
flag_carrier_before_approval       1359
dtype: int64

In [17]:
# 时间比较只在两端存在时进行；该表展示每条规则的适用范围。
orders_cleaning.summarize_rule_coverage(orders_v2, rules_version='v2')

,rule,scope,eligible_orders,outside_scope_orders,flagged_orders
0,flag_delivered_missing_date,状态为已交付,96478,2963,8
1,flag_delivered_missing_approval,状态为已交付,96478,2963,14
2,flag_delivered_missing_carrier,状态为已交付,96478,2963,2
3,flag_carrier_before_purchase,下单与交物流时间均有值,97658,1783,166
4,flag_delivery_before_carrier,交物流与送达时间均有值,96475,2966,23
5,flag_canceled_has_delivery_date,状态为取消,625,98816,6
6,flag_approval_before_purchase,下单与审批时间均有值,99281,160,0
7,flag_carrier_before_approval,审批与交物流时间均有值,97644,1797,1359


In [18]:
old_review = orders_v1['flag_needs_review']
new_rule_hit = orders_v2['flag_carrier_before_approval']
print('原清单订单数：', old_review.sum())
print('新规则触发订单数：', new_rule_hit.sum())
print('新规则与原清单重叠：', (new_rule_hit & old_review).sum())
print('新增关注订单：', (new_rule_hit & ~old_review).sum())
print('八规则核查并集：', orders_v2['flag_needs_review'].sum())
print('v1 延迟率：{:.2%}'.format(metrics_v1['late_rate']))
print('v2 延迟率：{:.2%}'.format(metrics_v2['late_rate']))

原清单订单数： 218
新规则触发订单数： 1359
新规则与原清单重叠： 166
新增关注订单： 1193
八规则核查并集： 1411
v1 延迟率：6.77%
v2 延迟率：6.77%


## 保存与练习

阶段报告已保存在 `reports/orders_v2/订单清洗报告_v2.md`。需要重新导出时，从项目目录运行：

```powershell
python .\src\02_clean_orders.py --rules-version v2
```

1. 为什么 218 笔原清单加上新规则命中 1,359 笔，合并后是 1,411 笔？

我的回答：因为有166例重合

2. 为什么核查范围扩大后，当前延迟率仍可保持一致？

我的回答：延迟判定直接比较实际送达日期与预计送达日期。本次新增核查标记没有改变已交付筛选和日期有效性等样本条件，因此分母仍为 96,470 笔，分子仍为 6,534 笔，延迟率保持 6.77%。


补充说明和下一课安排见 `docs/03_订单规则更新.md`。